# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.7 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 2880
Session ID: 3fbbeb07-0344-4592-a683-4670bdd0dcdc
Applying the following default arguments:
--glue_kernel_version 1.0.7
--enable-glue-datacatalog true
Waiting for session 3fbbeb07-0344-4592-a683-4670bdd0dcdc to get into ready status...
Session 3fbbeb07-0344-4592-a683-4670bdd0dcdc ha

In [2]:
dyf = glueContext.create_dynamic_frame_from_catalog(
    database='lds_raw',
    table_name='tarifa'
)

df = dyf.toDF()
df.show(10)
df.printSchema()


+---------+-------------+----------+--------------------+-------------+--------------------+------------+----------+-------------+--------+--------+---------------+-------------+---------------------+
|id_tarifa|codigo_tarifa|cod_tarifa|         descripcion|nivel_tension|   segmento_objetivo|tipo_cliente|cargo_fijo|cargo_energia|cargo_hp|cargo_fp|incluye_demanda|estado_tarifa|fecha_inicio_vigencia|
+---------+-------------+----------+--------------------+-------------+--------------------+------------+----------+-------------+--------+--------+---------------+-------------+---------------------+
|        1|          BT2|   BT2_COM|Tarifa comercial ...|           BT|COMERCIAL/OTROS/A...|   Comercial|       1.0|        0.715|     0.0|     0.0|              N|       ACTIVO|           2018-01-01|
|        2|          BT2|   BT2_GOB|Tarifa gobierno B...|           BT|            GOBIERNO|    Gobierno|       1.0|       0.6825|     0.0|     0.0|              N|       ACTIVO|           2018-01

In [3]:
total = df.count()
print("Total filas:", total)


Total filas: 40


In [4]:
import pyspark.sql.functions as F

print("=== NULOS POR COLUMNA ===")
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()


=== NULOS POR COLUMNA ===
+---------+-------------+----------+-----------+-------------+-----------------+------------+----------+-------------+--------+--------+---------------+-------------+---------------------+
|id_tarifa|codigo_tarifa|cod_tarifa|descripcion|nivel_tension|segmento_objetivo|tipo_cliente|cargo_fijo|cargo_energia|cargo_hp|cargo_fp|incluye_demanda|estado_tarifa|fecha_inicio_vigencia|
+---------+-------------+----------+-----------+-------------+-----------------+------------+----------+-------------+--------+--------+---------------+-------------+---------------------+
|        0|            0|         0|          0|            0|                0|           0|         0|            0|       0|       0|              0|            0|                    0|
+---------+-------------+----------+-----------+-------------+-----------------+------------+----------+-------------+--------+--------+---------------+-------------+---------------------+


In [5]:
string_cols = [c for c, t in df.dtypes if t == "string"]

print("=== BLANCOS POR COLUMNA ===")
df.select([
    F.count(F.when(F.col(c) == "", c)).alias(c)
    for c in string_cols
]).show()


=== BLANCOS POR COLUMNA ===
+-------------+----------+-----------+-------------+-----------------+------------+---------------+-------------+---------------------+
|codigo_tarifa|cod_tarifa|descripcion|nivel_tension|segmento_objetivo|tipo_cliente|incluye_demanda|estado_tarifa|fecha_inicio_vigencia|
+-------------+----------+-----------+-------------+-----------------+------------+---------------+-------------+---------------------+
|            0|         0|          0|            0|                0|           0|              0|            0|                    0|
+-------------+----------+-----------+-------------+-----------------+------------+---------------+-------------+---------------------+


In [8]:
df.groupBy("cod_tarifa").count().orderBy("count", ascending=False).show()


+----------+-----+
|cod_tarifa|count|
+----------+-----+
|   BT2_GOB|    1|
|   BT2_IND|    1|
|   BT2_RES|    1|
|   BT3_IND|    1|
|   BT4_GOB|    1|
|   BT5_GOB|    1|
|  BT5D_RES|    1|
|  BT5E_IND|    1|
|   MT2_GOB|    1|
|   MT3_GOB|    1|
|   MT3_RES|    1|
|   MT4_GOB|    1|
|   MT4_IND|    1|
|   MT4_RES|    1|
|   BT2_COM|    1|
|   BT3_COM|    1|
|   BT3_GOB|    1|
|   BT3_RES|    1|
|   BT5_RES|    1|
|  BT5A_COM|    1|
+----------+-----+
only showing top 20 rows


In [7]:
df.select(
    F.min("cargo_fijo").alias("min_cargo_fijo"),
    F.max("cargo_fijo").alias("max_cargo_fijo"),
    F.min("cargo_energia").alias("min_cargo_kwh"),
    F.max("cargo_energia").alias("max_cargo_kwh"),
).show()


+--------------+--------------+-------------+-------------+
|min_cargo_fijo|max_cargo_fijo|min_cargo_kwh|max_cargo_kwh|
+--------------+--------------+-------------+-------------+
|          0.98|           1.0|       0.4225|        0.715|
+--------------+--------------+-------------+-------------+
